In [184]:
# attempt to check how things are

In [4]:
import glyph_utils

In [6]:
glyph_utils.gardiner_to_hieroglyphics('Aa20')

'𓐢'

In [ ]:
glyph_utils.gard

In [1]:
import openai

In [2]:
system_prompt = """Assess the hieroglyphics translation based on the following scales and include your assessment in a JSON response:

1. **quality:**  
   - **1 (Poor):** Significant grammatical or logical errors.
   - **2 (Fair):** Generally correct but may need minor corrections or clarifications.
   - **3 (Good):** Fully reasonable seeming translation.

2. **interest:**  
   - **1 (Not Interesting):** Lacks novelty or fails to engage.
   - **2 (Somewhat Interesting):** Contains interesting elements but not compelling throughout.
   - **3 (Very Interesting):** Interesting OR sounds like ancient wisdom OR could be interpreted in a sexual way.

### **Instructions for the LLM:**  
- Respond only using the JSON object format specified above, without any additional text or comments.
- Evaluate both the **quality** and **interest level** of the translation according to the scales provided.
- Ensure the JSON output is correctly formatted and free of errors.

Example of the required JSON response:
{
  "quality": <int>,
  "interest": <int>
}"""


# blessing or a curse

In [42]:
glyph = "You can sit on the border of the river. The fields I gave to you, as you said."

In [16]:
ask_llm(glyph, system_prompt=system_prompt)


'{\n  "quality": 2,\n  "interest": 2\n}'

In [7]:
from openai import OpenAI  # Make sure to import the correct library


In [206]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise ValueError('OPENAI_API_KEY not found in environment or .env file')

def ask_llm(prompt, system_prompt="you are a helpful assistant", temp=0.8, model="gpt-40"):
    model_name = "gpt-4o"
    client = OpenAI(api_key=api_key)
    completion = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )

    return completion.choices[0].message.content


In [15]:
ask_llm(glyph)

'{\n  "quality": 3,\n  "interest": 2\n}'

In [187]:
from matviz.helpers_graphing import *
fnames = glob.glob("out/*.csv")

fnames

['out/translated_0.9.csv',
 'out/translated_0.8.csv',
 'out/translated_1.0.csv',
 'out/merged_translations.csv',
 'out/translated_0.6.csv',
 'out/translated_0.7.csv',
 'out/translated_0.5.csv']

In [201]:
df = pd.read_csv("out/merged_translations.csv")
df
# select only one translation to review
df = df.groupby('english').head(1)

In [199]:
len(df.groupby('english').head(1))

15825

In [200]:
len(set(df['glyph'])), len(set(df['english']))

(17446, 15826)

In [155]:
all_temps = ['0.9', '0.8', '1.0', '0.7']
all_temps = ['0.5', '0.6']

In [156]:
all_translations = {k[-7:-4]: pd.read_csv(k) for k in fnames}
# all_translations

In [157]:
all_translations.keys()

dict_keys(['0.9', '0.8', '1.0', '0.6', '0.7', '0.5'])

In [158]:
ii = 0
content_list = []
temps_list = []
glyph_list = []
gardiner_list = []
for cur_temp in all_temps:
    cur = all_translations[cur_temp]
    content_list += list(cur['english'].values)
    temps_list += [cur_temp] * len(cur)
    glyph_list += list(cur['glyph'].values)
    gardiner_list += list(cur['gardiner'].values)



In [159]:
{cur_temp: len(all_translations[cur_temp]) for cur_temp in all_temps}

{'0.5': 38, '0.6': 43}

In [193]:
content_list


['Priest of the Cheop, administrator of the royal property Nefer-nisut.',
 "The priest of the Sahura, the wab priest of the king, the priest of the Cheop, the priest of the Neferirkare, the priest of the Chentit, the served by the great God. The administrator of the king's property Nefer-bau-Ptah, the priest of his lord Dedet-tawi, whom the daughter of the daughter of the daughter of the daughter of the daughter of the daughter of the daughter of the daughter of the daughter.",
 "I performed the Maat, I performed the King of Upper and Lower Egypt Kheperkare, beloved of I. To my Lord, to the great God, Kai-em-nebef, one of his Lord's lovers, one of his Lord's lovers, Kai-em-anch.",
 'He will not hear.',
 "John's bread: 1, Fauliges-tree drugs: 1, beef fat: 1, wine: 1.",
 '1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 time 1 

In [203]:
content_list = list(df['english'].values)   

In [204]:
content_list

['Priest of the Cheop, administrator of the royal property and administrator of the royal property.',
 'It will be taken over 4 days.',
 'O predecessor of all predecessors in eternity,',
 'The righteous righteous righteous righteous righteous righteous righteous righteous righteous righteous righteous righteous You, the king, are in his body, and you say:',
 'It is useful for the servants of his abdomen according to his possessions.',
 'Blood from his nose.',
 "His mother, the administrator of the king's property, the guardian of the great God, the administrator of the king's property and the predecessor of the great good Nefer-nisut.",
 "John's bread: 1, sweet beer: 1.",
 'Is it in the presence of the hereditary noble and local prince, seal-bearer of the king? The only friend, overseer of troops, he says, Mehi, his name is Nen-Seschem-Zepter.',
 'Hereditary noble and local prince, seal-bearer of the king, the sole friend, overseer of the treasury of his neck, beloved one.',
 'On the o

In [207]:
import json

# Prepare the .jsonl file with each prompt as a request
with open("batch_input.jsonl", "w") as f:
    for i, content in enumerate(content_list):
        request = {
            "custom_id": f"request-{i+1}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "gpt-4o",
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": content}
                ],
                "max_tokens": 1000
            }
        }
        f.write(json.dumps(request) + "\n")


In [208]:
len(content_list)

15825

In [209]:
from openai import OpenAI

# Initialize the OpenAI client with your API key
client = OpenAI(api_key=api_key)

# Upload the input file to the Batch API
batch_input_file = client.files.create(
    file=open("batch_input.jsonl", "rb"),
    purpose="batch"
)



In [162]:
batch_input_file


FileObject(id='file-XHAtCPvqSpxXgURBCsj7JFge', bytes=110924, created_at=1729798506, filename='batch_input.jsonl', object='file', purpose='batch', status='processed', status_details=None)

In [210]:
batch_input_file


FileObject(id='file-s8AY1tFQ19HA2AyLtcWHw1ig', bytes=21172703, created_at=1729897159, filename='batch_input.jsonl', object='file', purpose='batch', status='processed', status_details=None)

In [211]:
batch_input_file_id = batch_input_file.id

client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
      "description": "big tester one"
    }
)

Batch(id='batch_671c22e67ce0819087b0722b5a2e82a0', completion_window='24h', created_at=1729897190, endpoint='/v1/chat/completions', input_file_id='file-s8AY1tFQ19HA2AyLtcWHw1ig', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1729983590, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'description': 'big tester one'}, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))

In [213]:
from openai import OpenAI
# client = OpenAI()

# tmp = client.batches.retrieve("batch_671594b3cf148190a5fe0c0e8e804866")
# tmp = client.batches.retrieve("batch_671aa17c67e88190934d50fb3531604d")
tmp = client.batches.retrieve("batch_671c22e67ce0819087b0722b5a2e82a0")
tmp

Batch(id='batch_671c22e67ce0819087b0722b5a2e82a0', completion_window='24h', created_at=1729897190, endpoint='/v1/chat/completions', input_file_id='file-s8AY1tFQ19HA2AyLtcWHw1ig', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1729903166, error_file_id=None, errors=None, expired_at=None, expires_at=1729983590, failed_at=None, finalizing_at=1729900819, in_progress_at=1729897196, metadata={'description': 'big tester one'}, output_file_id='file-XFOFAVergcWYJqqPEuo2IVD4', request_counts=BatchRequestCounts(completed=15825, failed=0, total=15825))

In [173]:
tmp.output_file_id

'file-3WvAe9qNzscyrWZYdUvUgHVQ'

In [215]:
from openai import OpenAI
# client = OpenAI()
client = OpenAI(api_key=api_key)
file_response2 = client.files.content(tmp.output_file_id)
print(file_response.text)

{"id": "batch_req_671598dae2188190a6e2f83d83a784a2", "custom_id": "request-1", "response": {"status_code": 200, "request_id": "8cd07714465761b53d15ea211b46e20a", "body": {"id": "chatcmpl-AKa2kesew2ewnNtwSaV5ratdutMnS", "object": "chat.completion", "created": 1729468474, "model": "gpt-4o-2024-08-06", "choices": [{"index": 0, "message": {"role": "assistant", "content": "```json\n{\n  \"quality\": 1,\n  \"interest\": 1\n}\n```", "refusal": null}, "logprobs": null, "finish_reason": "stop"}], "usage": {"prompt_tokens": 276, "completion_tokens": 20, "total_tokens": 296, "prompt_tokens_details": {"cached_tokens": 0}, "completion_tokens_details": {"reasoning_tokens": 0}}, "system_fingerprint": "fp_a7d06e42a7"}}, "error": null}
{"id": "batch_req_671598db118881908652a1061c9b6e3f", "custom_id": "request-2", "response": {"status_code": 200, "request_id": "f2927c2758e5e3c92c3f0cddbd35967b", "body": {"id": "chatcmpl-AKa1xU0W7UzVUnQ6kqqXIRVwE7qUh", "object": "chat.completion", "created": 1729468425, 

In [175]:
for w in file_response.text.split("\n")[-1]:
    print("trying")
    json.loads(w)

In [24]:
import json

,Unnamed: 0,gardiner,glyph,english
0,0,Aa1 G43 I9 G43 R8 U36 M23 X1 D21 Aa1 M23 X1 N3...,𓐍 𓅱 𓆑 𓅱 𓊹 𓍛 𓇓 𓏏 𓂋 𓐍 𓇓 𓏏 𓈖 𓊪 𓇋,"Priest of the Cheop, administrator of the roya..."
1,1,D46 D21 A24 D21 N5 4,𓂧 𓂋 𓀜 𓂋 𓇳 4,It will be taken over 4 days.
2,2,G17 D21 / / / / V31A Y1 G17 / N35 G21 V28 N5 V...,𓅓 𓂋 / / / / 𓎡A 𓏛 𓅓 / 𓈖 𓅘 𓎛 𓇳 𓎛 𓎛,"O predecessor of all predecessors in eternity,"
3,3,/ / / / / / / / / / U7 D21 X1 I9 G17 O34 N35 I...,/ / / / / / / / / / 𓌻 𓂋 𓏏 𓆑 𓅓 𓊃 𓈖 𓆑 / 𓄤 𓏏 [E14...,The righteous righteous righteous righteous ri...
4,4,G25 Aa1 Y1 N35 U31 X1 O1 A1 Z2 N35 X1 D21 F32 ...,𓅜 𓐍 𓏛 𓈖 𓍕 𓏏 𓉐 𓀀 𓏥 𓈖 𓏏 𓂋 𓄡 𓏏 𓏤 𓆑 𓂋 𓐍 𓏏 𓏛 𓏥,It is useful for the servants of his abdomen a...
...,...,...,...,...
17800,31,M17 Z7 G17 D36 V31A Z4 Y1 A1 W19 M17 O4 G1 D58...,𓇋 𓏲 𓅓 𓂝 𓎡A 𓏭 𓏛 𓀀 𓏇 𓇋 𓉔 𓄿 𓃀 𓂻 𓈖 𓀀 𓄖 𓅓 𓀀,My companion was like a flight to me.
17801,32,/ / O1 Z1 G17 D21 D34 G1 A24 A1 Z2 D21 D4 X1 Z...,/ / 𓉐 𓏤 𓅓 𓂋 𓂚 𓄿 𓀜 𓀀 𓏥 𓂋 𓁹 𓏏 𓏤 𓎡A,The house manager and the worfler in your eye.
17803,34,O34 Q3 O50 D1 Q3 Z4 N35 D21 M17 M17 X1 O1,𓊃 𓊪 𓊗 𓁶 𓊪 𓏭 𓈖 𓂋 𓇋 𓇋 𓏏 𓉐,The first time of the webery.
17804,35,S29 G36 D21 M17 N35A A2 Z7 S29 X1 N35 W24 Z7 T...,𓋴 𓅨 𓂋 𓇋 𓈖A 𓀁 𓏲 𓋴 𓏏 𓈖 𓏌 𓏲 𓌙 𓅯 𓏛 𓏥 𓈖 𓇋 𓏏 𓎛 𓏲 𓌙 𓏱...,The water of the Tamariske is drunk.


In [176]:
df = pd.DataFrame(np.concatenate([[temps_list], [content_list], [glyph_list], [gardiner_list]]), index=["temps", "content", "glyphs", "gardiner"]).T
# pd.DataFrame(response_list)


In [177]:
df

,temps,content,glyphs,gardiner
0,0.5,"Priest of the Cheop, administrator of the roya...",𓐍 𓅱 𓆑 𓅱 𓊹 𓍛 𓇓 𓏏 𓂋 𓐍 𓇓 𓏏 𓈖 𓄤,Aa1 G43 I9 G43 R8 U36 M23 X1 D21 Aa1 M23 X1 N3...
1,0.5,"The priest of the Sahura, the wab priest of th...",𓇳 𓃃 𓅱 𓊹 𓍛 𓇓 𓏏 𓈖 𓀆 𓐍 𓅱 𓆑 𓅱 𓊹 𓍛 𓇳 𓄤 𓁹 𓂓 𓊹 𓍛 𓏃 𓏏 ...,N5 D61 G43 R8 U36 M23 X1 N35 A6 Aa1 G43 I9 G43...
2,0.5,"I performed the Maat, I performed the King of ...",𓇋 𓅱 𓁹 𓈖 𓅱 𓌳 𓐙 𓂝 𓏏 𓁦 𓁹 𓈖 𓇓 𓏏 𓆤 𓏏 𓏏 𓇳 𓆣 𓂓 𓌻 𓂋 𓇋 ...,M17 G43 D4 N35 G43 U1 Aa11 D36 X1 C10 D4 N35 M...
3,0.5,He will not hear.,𓂜 𓈖 𓄔 𓅓 𓈖 𓆑 𓏏 𓏲,D35 N35 F21 G17 N35 I9 X1 Z7
4,0.5,"John's bread: 1, Fauliges-tree drugs: 1, beef ...",𓇋 𓏲 𓈖 𓍑 𓄿 𓂋 𓏏 𓈒 𓏥 𓏤 𓆱 𓂝 𓍯 𓄿 𓈒 𓏥 𓏤 𓎙 𓂧 𓏌 𓏥 𓃒 𓏤 ...,M17 Z7 N35 U28 G1 D21 X1 N33 Z2 Z1 M3 D36 V4 G...
...,...,...,...,...
76,0.6,"(According to the Qur’an, the Qur’an is the Qu...",/ / / / / / / / / / / / / / / / / / / / / / / / /,/ / / / / / / / / / / / / / / / / / / / / / / / /
77,0.6,I was in the residence of the one who was in h...,𓈖 𓇦 𓏲 𓀀 𓅓 𓄚 𓈖 𓏌 𓏲 𓉐 𓈖 𓈖 𓏏 𓏭 𓍋 𓅓 𓂋 𓉴 𓉐 𓆑,N35 M37 Z7 A1 G17 F26 N35 W24 Z7 O1 N35 N35 X1...
78,0.6,What to do:,𓁹 𓂋 𓏏 𓏥 𓂋 𓊃,D4 D21 X1 Z2 D21 O34
79,0.6,Don’t get closer to the stomach to remove from...,𓅓 𓎡A 𓏏 𓏴 𓂻 𓏲 𓁷 𓏤 𓈖 𓏏 𓏭 𓂋 𓏤 𓄣 𓏤 𓂧 𓂋 𓀜 𓅓 𓂋 𓏲 𓏏 𓏭 𓇯,G17 V31A X1 Z9 D54 Z7 D2 Z1 N35 X1 Z4 D21 Z1 F...


In [133]:
len(temps_list), len(content_list), len(response_list), len(glyph_list), len(gardiner_list)

(217, 217, 217, 217, 217)

In [32]:
cur_res = response_list[0]['response']['body']['choices'][0]['message']['content']

In [225]:
df[all_txt=="I'm sorry, it seems there is a part of the content missing for me to evaluate. Could you provide the hieroglyphics translation text that needs to be assessed?"]

,Unnamed: 0,gardiner,glyph,english
395,395,G17 D36,𓅓 𓂝,It is


In [223]:
all_txt = np.array([cur_res['response']['body']['choices'][0]['message']['content'] for cur_res in response_list])

In [226]:
cur_res[]

{'id': 'batch_req_671c314b18988190aaef378126495d92',
 'custom_id': 'request-382',
 'response': {'status_code': 200,
  'request_id': 'baddf8b09287aa676ba0c5452c2b6921',
  'body': {'id': 'chatcmpl-AMNbCPQ6MDZ1bFJvwddzfxiIYCpmY',
   'object': 'chat.completion',
   'created': 1729897294,
   'model': 'gpt-4o-2024-08-06',
   'choices': [{'index': 0,
     'message': {'role': 'assistant',
      'content': "I'm sorry, it seems there is a part of the content missing for me to evaluate. Could you provide the hieroglyphics translation text that needs to be assessed?",
      'refusal': None},
     'logprobs': None,
     'finish_reason': 'stop'}],
   'usage': {'prompt_tokens': 235,
    'completion_tokens': 34,
    'total_tokens': 269,
    'prompt_tokens_details': {'cached_tokens': 0},
    'completion_tokens_details': {'reasoning_tokens': 0}},
   'system_fingerprint': 'fp_90354628f2'}},
 'error': None}

In [240]:
cur_txt
cur_json

{'quality': 2, 'interest': 2}

In [242]:
response_list = [json.loads(w) for w in file_response2.text.split("\n")[:-1]]
all_res = []
for cur_res in response_list:
    cur_txt = cur_res['response']['body']['choices'][0]['message']['content']
    try:
        cur_json = robust_json_loader(cur_txt)
    except:
        if cur_txt == "I'm sorry, but I cannot accurately evaluate the quality and interest of the translation without the hieroglyphics or their translated text. Could you please provide the translation for assessment?":
            cur_json = {'quality': 2, 'interest': 2}
        else:
            print(f"text is:`{cur_txt}`")
            print(df[all_txt==cur_txt]['glyph'])
            print(df[all_txt==cur_txt]['english'])
            print("")
            cur_json = {'quality': -1, 'interest': -1}
    all_res.append(cur_json)

df_all2 = pd.concat([df, pd.DataFrame(all_res)], axis=1)
df_all2

text is:`I'm sorry, it seems there is a part of the content missing for me to evaluate. Could you provide the hieroglyphics translation text that needs to be assessed?`
395    𓅓 𓂝
Name: glyph, dtype: object
395    It is
Name: english, dtype: object

text is:`I'm sorry, I can't assist with that request.`
2371    / / / / / 𓅓 𓊖 𓏏 𓏤
Name: glyph, dtype: object
2371    (According to the Qur’an, it is
Name: english, dtype: object

text is:`I'm sorry, but I need you to provide the hieroglyphics translation for assessment. Once you do, I'll be able to evaluate it based on the specified criteria.`
3419    𓇋 𓅱 𓁹 𓈖
Name: glyph, dtype: object
3419    I made
Name: english, dtype: object

text is:`I'm sorry, but I need more information about the hieroglyphics or the translation in order to provide an evaluation. Could you please provide the text or the content of the hieroglyphics in need of assessment?`
4049    𓁷 𓏤 𓊪 𓏲
Name: glyph, dtype: object
4049    This means
Name: english, dtype: object

text 

,Unnamed: 0,gardiner,glyph,english,quality,interest
0,0.0,Aa1 G43 I9 G43 R8 U36 M23 X1 D21 Aa1 M23 X1 N3...,𓐍 𓅱 𓆑 𓅱 𓊹 𓍛 𓇓 𓏏 𓂋 𓐍 𓇓 𓏏 𓈖 𓊪 𓇋,"Priest of the Cheop, administrator of the roya...",2.0,2.0
1,1.0,D46 D21 A24 D21 N5 4,𓂧 𓂋 𓀜 𓂋 𓇳 4,It will be taken over 4 days.,1.0,1.0
2,2.0,G17 D21 / / / / V31A Y1 G17 / N35 G21 V28 N5 V...,𓅓 𓂋 / / / / 𓎡A 𓏛 𓅓 / 𓈖 𓅘 𓎛 𓇳 𓎛 𓎛,"O predecessor of all predecessors in eternity,",3.0,3.0
3,3.0,/ / / / / / / / / / U7 D21 X1 I9 G17 O34 N35 I...,/ / / / / / / / / / 𓌻 𓂋 𓏏 𓆑 𓅓 𓊃 𓈖 𓆑 / 𓄤 𓏏 [E14...,The righteous righteous righteous righteous ri...,1.0,1.0
4,4.0,G25 Aa1 Y1 N35 U31 X1 O1 A1 Z2 N35 X1 D21 F32 ...,𓅜 𓐍 𓏛 𓈖 𓍕 𓏏 𓉐 𓀀 𓏥 𓈖 𓏏 𓂋 𓄡 𓏏 𓏤 𓆑 𓂋 𓐍 𓏏 𓏛 𓏥,It is useful for the servants of his abdomen a...,2.0,2.0
...,...,...,...,...,...,...
15766,NaN,NaN,NaN,NaN,2.0,3.0
15788,NaN,NaN,NaN,NaN,1.0,1.0
15792,NaN,NaN,NaN,NaN,2.0,2.0
15802,NaN,NaN,NaN,NaN,1.0,2.0


In [255]:
pd.DataFrame(all_res)




,quality,interest
0,2,2
1,1,1
2,3,3
3,1,1
4,2,2
...,...,...
15820,3,2
15821,1,2
15822,1,2
15823,2,2


In [263]:
df_res = pd.DataFrame(all_res)

df_all = copy.deepcopy(df)

df_all['quality'] = df_res['quality']
df_all['interest'] = df_res['interest']
df_all

,Unnamed: 0,gardiner,glyph,english,quality,interest
0,0,Aa1 G43 I9 G43 R8 U36 M23 X1 D21 Aa1 M23 X1 N3...,𓐍 𓅱 𓆑 𓅱 𓊹 𓍛 𓇓 𓏏 𓂋 𓐍 𓇓 𓏏 𓈖 𓊪 𓇋,"Priest of the Cheop, administrator of the roya...",2.0,2.0
1,1,D46 D21 A24 D21 N5 4,𓂧 𓂋 𓀜 𓂋 𓇳 4,It will be taken over 4 days.,1.0,1.0
2,2,G17 D21 / / / / V31A Y1 G17 / N35 G21 V28 N5 V...,𓅓 𓂋 / / / / 𓎡A 𓏛 𓅓 / 𓈖 𓅘 𓎛 𓇳 𓎛 𓎛,"O predecessor of all predecessors in eternity,",3.0,3.0
3,3,/ / / / / / / / / / U7 D21 X1 I9 G17 O34 N35 I...,/ / / / / / / / / / 𓌻 𓂋 𓏏 𓆑 𓅓 𓊃 𓈖 𓆑 / 𓄤 𓏏 [E14...,The righteous righteous righteous righteous ri...,1.0,1.0
4,4,G25 Aa1 Y1 N35 U31 X1 O1 A1 Z2 N35 X1 D21 F32 ...,𓅜 𓐍 𓏛 𓈖 𓍕 𓏏 𓉐 𓀀 𓏥 𓈖 𓏏 𓂋 𓄡 𓏏 𓏤 𓆑 𓂋 𓐍 𓏏 𓏛 𓏥,It is useful for the servants of his abdomen a...,2.0,2.0
...,...,...,...,...,...,...
17800,31,M17 Z7 G17 D36 V31A Z4 Y1 A1 W19 M17 O4 G1 D58...,𓇋 𓏲 𓅓 𓂝 𓎡A 𓏭 𓏛 𓀀 𓏇 𓇋 𓉔 𓄿 𓃀 𓂻 𓈖 𓀀 𓄖 𓅓 𓀀,My companion was like a flight to me.,NaN,NaN
17801,32,/ / O1 Z1 G17 D21 D34 G1 A24 A1 Z2 D21 D4 X1 Z...,/ / 𓉐 𓏤 𓅓 𓂋 𓂚 𓄿 𓀜 𓀀 𓏥 𓂋 𓁹 𓏏 𓏤 𓎡A,The house manager and the worfler in your eye.,NaN,NaN
17803,34,O34 Q3 O50 D1 Q3 Z4 N35 D21 M17 M17 X1 O1,𓊃 𓊪 𓊗 𓁶 𓊪 𓏭 𓈖 𓂋 𓇋 𓇋 𓏏 𓉐,The first time of the webery.,NaN,NaN
17804,35,S29 G36 D21 M17 N35A A2 Z7 S29 X1 N35 W24 Z7 T...,𓋴 𓅨 𓂋 𓇋 𓈖A 𓀁 𓏲 𓋴 𓏏 𓈖 𓏌 𓏲 𓌙 𓅯 𓏛 𓏥 𓈖 𓇋 𓏏 𓎛 𓏲 𓌙 𓏱...,The water of the Tamariske is drunk.,NaN,NaN


In [275]:
# select where inteest is 3 and quality = 3
df_all['interest'] == 3 
df_all['quality'] == 3
the_best = df_all[(df_all['interest'] == 3) & (df_all['quality'] == 3)]


In [300]:
df_all.to_csv("all_readglyphs.csv")

In [295]:
# search for - blessing, curse, 

,Unnamed: 0,gardiner,glyph,english,quality,interest
2,2,G17 D21 / / / / V31A Y1 G17 / N35 G21 V28 N5 V...,𓅓 𓂋 / / / / 𓎡A 𓏛 𓅓 / 𓈖 𓅘 𓎛 𓇳 𓎛 𓎛,"O predecessor of all predecessors in eternity,",3.0,3.0
39,39,D2 Z1 D2 Z1 N35 X1 X1 U15 G17 A40 D36 N35 I9 D...,𓁷 𓏤 𓁷 𓏤 𓈖 𓏏 𓏏 𓍃 𓅓 𓀭 𓂝 𓈖 𓆑 𓂧 𓏏 𓏤 𓆑,"On the face of Atum, he placed his hand.",3.0,3.0
50,50,/ / D21 D36 N5 G7 D2 Z1 G7 M17 A2 D21 M17 A2 X...,/ / 𓂋 𓂝 𓇳 𓅆 𓁷 𓏤 𓅆 𓇋 𓀁 𓂋 𓇋 𓀁 𓏏 𓏲 𓆑 𓅓 𓅯 𓄿 𓅱 𓏏 𓉔 ...,"Re said to him, It is in the shadow of the sha...",3.0,3.0
55,55,M17 D21 F13 Q3 Z9 V31A O29 X1 H8 N35 X1 V26 D4...,𓇋 𓂋 𓄋 𓊪 𓏴 𓎡A 𓉻 𓏏 𓆇 𓈖 𓏏 𓎙 𓂧 𓏌 𓏥 𓅨 𓂋 𓂧 𓏲 𓀉 𓅪 / /...,"If you find a.t swallow of fat, then you shoul...",3.0,3.0
60,60,D21 Q3 D36 F4 D36 L2 X1 S19 S29 U23 T21 X1 Z1 ...,𓂋 𓊪 𓂝 𓄂 𓂝 𓆤 𓏏 𓋨 𓋴 𓍋 𓌡 𓏏 𓏤 𓄓 𓏏 𓎼 𓎼 𓎼 𓎛 𓋴,"Hereditary noble and local prince, seal-bearer...",3.0,3.0
...,...,...,...,...,...,...
15755,7347,M17 O34 O34 O34 M17 / I9 Aa11 X1 P8h,𓇋 𓊃 𓊃 𓊃 𓇋 / 𓆑 𓐙 𓏏 𓊤h,"His wife, justified.",3.0,3.0
15776,7368,G39 X1 I9 U7 X1 I9 W17D N35 X1 M17 X1,𓅭 𓏏 𓆑 𓌻 𓏏 𓆑 𓏃D 𓈖 𓏏 𓇋 𓏏,His beloved daughter Chentit.,3.0,3.0
15780,7372,S29 T3 M12 Aa1 M33 M39 O4 O29 M15 O1 M155 O29 ...,𓋴 𓌉 𓆼 𓐍 𓇠 𓇨 𓉔 𓉻 𓇇 𓉐 [M155] 𓉻 𓆼 /,The supervision of a thousand grapes and grape...,3.0,3.0
15803,7395,F4 D36 G17 S29 Aa1 Z7 A2 Z3A V31A X1 F42 D21 D...,𓄂 𓂝 𓅓 𓋴 𓐍 𓏲 𓀁 𓏪A 𓎡A 𓏏 𓄭 𓂋 𓂻 𓈖 𓁹 𓏏 𓏤 𓂋 𓇋 𓅱 𓂋 𓏥 ...,The beginning of the complaints about the reac...,3.0,3.0


In [298]:
the_best.iloc[-2]['english']

'The beginning of the complaints about the reach of the eye to the place of the abdomen: a disease that I will treat.'

In [290]:

the_best[['glyph', 'english']]

# loop to print the first several
for i, row in the_best.iterrows():
    print(f"Glyph: {row['glyph']}\n {row['english']}\n \n")

Glyph: 𓅓 𓂋 / / / / 𓎡A 𓏛 𓅓 / 𓈖 𓅘 𓎛 𓇳 𓎛 𓎛
 O predecessor of all predecessors in eternity,
 

Glyph: 𓁷 𓏤 𓁷 𓏤 𓈖 𓏏 𓏏 𓍃 𓅓 𓀭 𓂝 𓈖 𓆑 𓂧 𓏏 𓏤 𓆑
 On the face of Atum, he placed his hand.
 

Glyph: / / 𓂋 𓂝 𓇳 𓅆 𓁷 𓏤 𓅆 𓇋 𓀁 𓂋 𓇋 𓀁 𓏏 𓏲 𓆑 𓅓 𓅯 𓄿 𓅱 𓏏 𓉔 𓃀 / /
 Re said to him, It is in the shadow of the shadow.
 

Glyph: 𓇋 𓂋 𓄋 𓊪 𓏴 𓎡A 𓉻 𓏏 𓆇 𓈖 𓏏 𓎙 𓂧 𓏌 𓏥 𓅨 𓂋 𓂧 𓏲 𓀉 𓅪 / / / / 𓏇 𓇋 𓈖 𓏏 𓏭 𓂜 𓈖 𓃹 𓈖 𓄥 𓄿 𓅱 𓏏 𓐎 𓏥 𓅓 𓂝 𓏏 𓄹 𓏥 𓎟 𓏏
 If you find a.t swallow of fat, then you should say: It is the case that there is no blame in any part of the body.
 

Glyph: 𓂋 𓊪 𓂝 𓄂 𓂝 𓆤 𓏏 𓋨 𓋴 𓍋 𓌡 𓏏 𓏤 𓄓 𓏏 𓎼 𓎼 𓎼 𓎛 𓋴
 Hereditary noble and local prince, seal-bearer of the king, the sole friend, overseer of the treasury, Gehes.
 

Glyph: 𓅭 𓈖 𓇓 𓏏 𓈖 𓀆 𓅓 𓂋 𓉐 𓉐 𓉟 𓉐 𓉐 𓆑 𓅓 𓂋 𓏃 𓈙 𓏤 𓈉 𓇋 𓌳 
 The son of the wab priest of the king, the predecessor of his great good, the predecessor of the Chentuschi Imi.
 

Glyph: 𓇋 𓋴 𓎡A 𓇋 𓄿 𓂧 𓏏 𓈇 𓏤 𓎡A 𓂋 𓇥 𓂋 𓈖 𓏏 𓁶 𓏤 𓎡A
 Your bar is at the top of your head.
 

Glyph: 𓇋 𓂋 𓂡 𓎡A 𓊃 𓀀 𓏤 𓈖 𓄿 𓈖 𓃀 𓏲 𓎟 𓀀 𓁐 𓏥 𓈉 𓏏 𓏤 𓏥 𓂋 𓅯 𓄿 𓇋 𓇋 𓏏 𓂝 𓏏 𓉐 𓏪A 𓈖

In [276]:
prompt = """We need to review this heireoglpyhic translation. How cool is it on a scale from 1 to 10?"""


In [279]:
for ii in range(10):

    cur_txt = the_best['english'].iloc[ii]

    print(ask_llm(cur_txt, system_prompt=prompt))


I'd rate that an 8 out of 10 for coolness! The phrase "O predecessor of all predecessors in eternity" evokes a sense of ancient wisdom and timelessness, which aligns well with the mysterious and historic allure of hieroglyphics. It captures the imagination and invites one to ponder the vast expanse of history and the legacy of ancient civilizations.
The concept of "coolness" is subjective and can vary greatly depending on personal interests and cultural perspectives. However, interpreting or translating ancient hieroglyphs, such as those related to Egyptian mythology, can be quite fascinating and valuable from a historical and cultural standpoint. If you appreciate ancient history, mythology, and the art of deciphering ancient languages, you might rate this a higher number on the coolness scale, possibly a 7 or 8. For others less interested in these topics, it might rank lower. Ultimately, it depends on one's appreciation for ancient cultures and their mysteries.
On a scale from 1 to 1

In [281]:
the_best['english'].iloc[:10].values

array(['O predecessor of all predecessors in eternity,',
       'On the face of Atum, he placed his hand.',
       'Re said to him, It is in the shadow of the shadow.',
       'If you find a.t swallow of fat, then you should say: It is the case that there is no blame in any part of the body.',
       'Hereditary noble and local prince, seal-bearer of the king, the sole friend, overseer of the treasury, Gehes.',
       'The son of the wab priest of the king, the predecessor of his great good, the predecessor of the Chentuschi Imi.',
       'Your bar is at the top of your head.',
       'If you look at a man from the district of the foreign countries to the district of what is in this district, then you should say:',
       'See, there is no beef.',
       'The dead, the dead, the dead, the dead, the dead, the dead. The end of the end of the end of the end of the end.'],
      dtype=object)

In [282]:
df

,Unnamed: 0,gardiner,glyph,english
0,0,Aa1 G43 I9 G43 R8 U36 M23 X1 D21 Aa1 M23 X1 N3...,𓐍 𓅱 𓆑 𓅱 𓊹 𓍛 𓇓 𓏏 𓂋 𓐍 𓇓 𓏏 𓈖 𓊪 𓇋,"Priest of the Cheop, administrator of the roya..."
1,1,D46 D21 A24 D21 N5 4,𓂧 𓂋 𓀜 𓂋 𓇳 4,It will be taken over 4 days.
2,2,G17 D21 / / / / V31A Y1 G17 / N35 G21 V28 N5 V...,𓅓 𓂋 / / / / 𓎡A 𓏛 𓅓 / 𓈖 𓅘 𓎛 𓇳 𓎛 𓎛,"O predecessor of all predecessors in eternity,"
3,3,/ / / / / / / / / / U7 D21 X1 I9 G17 O34 N35 I...,/ / / / / / / / / / 𓌻 𓂋 𓏏 𓆑 𓅓 𓊃 𓈖 𓆑 / 𓄤 𓏏 [E14...,The righteous righteous righteous righteous ri...
4,4,G25 Aa1 Y1 N35 U31 X1 O1 A1 Z2 N35 X1 D21 F32 ...,𓅜 𓐍 𓏛 𓈖 𓍕 𓏏 𓉐 𓀀 𓏥 𓈖 𓏏 𓂋 𓄡 𓏏 𓏤 𓆑 𓂋 𓐍 𓏏 𓏛 𓏥,It is useful for the servants of his abdomen a...
...,...,...,...,...
17800,31,M17 Z7 G17 D36 V31A Z4 Y1 A1 W19 M17 O4 G1 D58...,𓇋 𓏲 𓅓 𓂝 𓎡A 𓏭 𓏛 𓀀 𓏇 𓇋 𓉔 𓄿 𓃀 𓂻 𓈖 𓀀 𓄖 𓅓 𓀀,My companion was like a flight to me.
17801,32,/ / O1 Z1 G17 D21 D34 G1 A24 A1 Z2 D21 D4 X1 Z...,/ / 𓉐 𓏤 𓅓 𓂋 𓂚 𓄿 𓀜 𓀀 𓏥 𓂋 𓁹 𓏏 𓏤 𓎡A,The house manager and the worfler in your eye.
17803,34,O34 Q3 O50 D1 Q3 Z4 N35 D21 M17 M17 X1 O1,𓊃 𓊪 𓊗 𓁶 𓊪 𓏭 𓈖 𓂋 𓇋 𓇋 𓏏 𓉐,The first time of the webery.
17804,35,S29 G36 D21 M17 N35A A2 Z7 S29 X1 N35 W24 Z7 T...,𓋴 𓅨 𓂋 𓇋 𓈖A 𓀁 𓏲 𓋴 𓏏 𓈖 𓏌 𓏲 𓌙 𓅯 𓏛 𓏥 𓈖 𓇋 𓏏 𓎛 𓏲 𓌙 𓏱...,The water of the Tamariske is drunk.


In [266]:
most_common(df_all2['quality'])

[(2.0, 5626),
 (1.0, 5309),
 (3.0, 4881),
 (-1.0, 9),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),
 (nan, 1),


In [111]:
df_all.groupby('temps')['quality'].mean()

temps
0.7    2.062500
0.8    1.963636
0.9    1.981818
1.0    2.152542
Name: quality, dtype: float64

In [125]:
df_all.groupby('temps')['interest'].mean()

temps
0.7    2.083333
0.8    1.836364
0.9    1.854545
1.0    1.966102
Name: interest, dtype: float64

In [142]:
df_all.to_csv("all_results.csv", index=False)

In [292]:
df_all

,Unnamed: 0,gardiner,glyph,english,quality,interest
0,0,Aa1 G43 I9 G43 R8 U36 M23 X1 D21 Aa1 M23 X1 N3...,𓐍 𓅱 𓆑 𓅱 𓊹 𓍛 𓇓 𓏏 𓂋 𓐍 𓇓 𓏏 𓈖 𓊪 𓇋,"Priest of the Cheop, administrator of the roya...",2.0,2.0
1,1,D46 D21 A24 D21 N5 4,𓂧 𓂋 𓀜 𓂋 𓇳 4,It will be taken over 4 days.,1.0,1.0
2,2,G17 D21 / / / / V31A Y1 G17 / N35 G21 V28 N5 V...,𓅓 𓂋 / / / / 𓎡A 𓏛 𓅓 / 𓈖 𓅘 𓎛 𓇳 𓎛 𓎛,"O predecessor of all predecessors in eternity,",3.0,3.0
3,3,/ / / / / / / / / / U7 D21 X1 I9 G17 O34 N35 I...,/ / / / / / / / / / 𓌻 𓂋 𓏏 𓆑 𓅓 𓊃 𓈖 𓆑 / 𓄤 𓏏 [E14...,The righteous righteous righteous righteous ri...,1.0,1.0
4,4,G25 Aa1 Y1 N35 U31 X1 O1 A1 Z2 N35 X1 D21 F32 ...,𓅜 𓐍 𓏛 𓈖 𓍕 𓏏 𓉐 𓀀 𓏥 𓈖 𓏏 𓂋 𓄡 𓏏 𓏤 𓆑 𓂋 𓐍 𓏏 𓏛 𓏥,It is useful for the servants of his abdomen a...,2.0,2.0
...,...,...,...,...,...,...
17800,31,M17 Z7 G17 D36 V31A Z4 Y1 A1 W19 M17 O4 G1 D58...,𓇋 𓏲 𓅓 𓂝 𓎡A 𓏭 𓏛 𓀀 𓏇 𓇋 𓉔 𓄿 𓃀 𓂻 𓈖 𓀀 𓄖 𓅓 𓀀,My companion was like a flight to me.,NaN,NaN
17801,32,/ / O1 Z1 G17 D21 D34 G1 A24 A1 Z2 D21 D4 X1 Z...,/ / 𓉐 𓏤 𓅓 𓂋 𓂚 𓄿 𓀜 𓀀 𓏥 𓂋 𓁹 𓏏 𓏤 𓎡A,The house manager and the worfler in your eye.,NaN,NaN
17803,34,O34 Q3 O50 D1 Q3 Z4 N35 D21 M17 M17 X1 O1,𓊃 𓊪 𓊗 𓁶 𓊪 𓏭 𓈖 𓂋 𓇋 𓇋 𓏏 𓉐,The first time of the webery.,NaN,NaN
17804,35,S29 G36 D21 M17 N35A A2 Z7 S29 X1 N35 W24 Z7 T...,𓋴 𓅨 𓂋 𓇋 𓈖A 𓀁 𓏲 𓋴 𓏏 𓈖 𓏌 𓏲 𓌙 𓅯 𓏛 𓏥 𓈖 𓇋 𓏏 𓎛 𓏲 𓌙 𓏱...,The water of the Tamariske is drunk.,NaN,NaN


In [294]:
df_all[df_all['interest'] == 3]

,Unnamed: 0,gardiner,glyph,english,quality,interest
2,2,G17 D21 / / / / V31A Y1 G17 / N35 G21 V28 N5 V...,𓅓 𓂋 / / / / 𓎡A 𓏛 𓅓 / 𓈖 𓅘 𓎛 𓇳 𓎛 𓎛,"O predecessor of all predecessors in eternity,",3.0,3.0
9,9,D21 Q3 D36 F4 D36 L2 X1 S19 S29 U23 T21 X1 Z1 ...,𓂋 𓊪 𓂝 𓄂 𓂝 𓆤 𓏏 𓋨 𓋴 𓍋 𓌡 𓏏 𓏤 𓅓 𓂋 𓋨 𓏏 𓈖 𓅘 𓎛 𓎛 𓃀 𓏏 ...,"Hereditary noble and local prince, seal-bearer...",2.0,3.0
26,26,V28 M2 N35 M2 N35 X1 H8 N35 X1 D21 M17 M17 X1 ...,𓎛 𓆰 𓈖 𓆰 𓈖 𓏏 𓆇 𓈖 𓏏 𓂋 𓇋 𓇋 𓏏 𓐎 𓏥 𓅓 𓂾 𓆑 𓏭 𓈇 𓏤 𓂋 𓂝 ...,The ḥnḥn.t swallow of the ry.t swallow on his ...,2.0,3.0
39,39,D2 Z1 D2 Z1 N35 X1 X1 U15 G17 A40 D36 N35 I9 D...,𓁷 𓏤 𓁷 𓏤 𓈖 𓏏 𓏏 𓍃 𓅓 𓀭 𓂝 𓈖 𓆑 𓂧 𓏏 𓏤 𓆑,"On the face of Atum, he placed his hand.",3.0,3.0
42,42,M17 Z7 I9 F51 Z2 I10 D46 M17 N35 V31A D21 O34,𓇋 𓏲 𓆑 𓄹 𓏥 𓆓 𓂧 𓇋 𓈖 𓎡A 𓂋 𓊃,Your flesh says about it:,2.0,3.0
...,...,...,...,...,...,...
15800,7392,D21 S29 D28 U13 D36 D2 Z1 U1 Aa11 D36 3 W24 Z1,𓂋 𓋴 𓂓 𓍁 𓂝 𓁷 𓏤 𓌳 𓐙 𓂝 3 𓏌 𓏤,It will be scattered by the third measure.,2.0,3.0
15801,7393,G36 D21 V28 W24 D36 W10 D1 M17 N35 D21 Q3 M43 ...,𓅨 𓂋 𓎛 𓏌 𓂝 𓎺 𓁶 𓇋 𓈖 𓂋 𓊪 𓇭 𓅱 [W79] 𓊹 𓍛 𓄤 𓂋 𓏏 𓇋 𓅓 ...,"The head of the Jnp, the priest of Heliopolis ...",2.0,3.0
15803,7395,F4 D36 G17 S29 Aa1 Z7 A2 Z3A V31A X1 F42 D21 D...,𓄂 𓂝 𓅓 𓋴 𓐍 𓏲 𓀁 𓏪A 𓎡A 𓏏 𓄭 𓂋 𓂻 𓈖 𓁹 𓏏 𓏤 𓂋 𓇋 𓅱 𓂋 𓏥 ...,The beginning of the complaints about the reac...,3.0,3.0
15805,7397,M17 D21 F32 X1 Z1 Q3 Z7 T28 D21 M17 M4 A50 Y1 ...,𓇋 𓂋 𓄡 𓏏 𓏤 𓊪 𓏲 𓌨 𓂋 𓇋 𓆳 𓀻 𓏛 𓈖 𓀀 𓐪 𓂧 𓏌 𓏛 𓀜 𓀀 𓏥 𓀗 ...,When it comes to that a generation belongs to ...,2.0,3.0


In [31]:
robust_json_loader = lambda cur_res: json.loads(cur_res.replace("```json", "").replace("```",""))

In [116]:
json.loads(cur_res.replace("```", ""))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [101]:
len(tmp)

55

In [103]:
len(content_list)

55

In [87]:
tmp = file_response.text.split("\n")

In [99]:
file_response.text.split("\n")[-1]

''

In [18]:
batch_input_file_id = "file-nrCvdtE0zR78MvPh5JZ3VjAh"

In [59]:

batch_input_file_id = batch_input_file.id
print(f"Uploaded batch file with ID: {batch_input_file_id}")

Uploaded batch file with ID: file-nrCvdtE0zR78MvPh5JZ3VjAh


In [61]:
batch_id = batch_input_file.id

In [62]:
# import time

# while True:
batch_status = client.batches.retrieve(batch_id)
status = batch_status["status"]
print(f"Batch status: {status}")

if status in ["completed", "failed", "expired"]:
    break

# time.sleep(10)  # Wait for 10 seconds before checking again


BadRequestError: Error code: 400 - {'error': {'message': "Invalid 'batch_id': 'file-nrCvdtE0zR78MvPh5JZ3VjAh'. Expected an ID that begins with 'batch'.", 'type': 'invalid_request_error', 'param': 'batch_id', 'code': 'invalid_value'}}